# 01 · Clean ResNet18 baseline on CIFAR-100

Research: **How Far Does Corruption Robustness Transfer? A Leave-One-Family-Out
Study on CIFAR-100-C**. This notebook implements only the clean control condition.

**Default: a two-epoch smoke test.** Its small-subset results check software and
must never appear as scientific findings. Full training requires changing
`SMOKE_TEST` below. The official clean test has a separate, opt-in final cell.
No CIFAR-100-C data is downloaded or evaluated.

In Colab choose **Runtime → Change runtime type → GPU** (a T4 is sufficient), then
run the cells in order. This notebook is self-contained: source-writing cells
below materialize the exact implementation into a temporary runtime package.
No repository clone or automatic package installation is required.


In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # Before importing torch/CUDA.
try:
    import torch
    import torchvision
    import numpy as np
    import matplotlib
except (ImportError, OSError, RuntimeError) as exc:
    raise RuntimeError(
        "A compatible torch/torchvision pair, NumPy, and Matplotlib are required. "
        "Use a fresh Colab runtime, or install requirements.txt locally. "
        "For binary operator errors reinstall torch and torchvision as a matched pair "
        "using https://pytorch.org/get-started/locally/ and restart the runtime."
    ) from exc
if not hasattr(torch, "amp") or not hasattr(torch.amp, "GradScaler"):
    raise RuntimeError("This notebook requires PyTorch >= 2.3. Update the runtime and restart.")
print("Python / packages imported successfully.")
print("PyTorch:", torch.__version__, "Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available(), "CUDA build:", torch.version.cuda)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}; VRAM: {gpu.total_memory / 2**30:.1f} GiB")
elif torch.backends.mps.is_available():
    print("Apple MPS available. Automatic device selection prefers MPS over CPU.")
else:
    print("CPU mode: suitable for a smoke test. Enable a GPU before full training.")


## Experiment settings — edit here

The full schedule is predefined: SGD (LR 0.1, momentum 0.9, weight decay 0.0005),
batch 128, 100 epochs, cosine decay to zero, ordinary cross entropy, seed 42.
These are starting settings, with no claim of optimality. The smoke schedule is
two epochs over **200 training / 100 validation** images, balanced over all classes.
The cosine `T_max` always equals the actual planned epochs for the selected mode.

Use the **same `SPLIT_SEED` and saved split** across later experimental conditions.
Keep `RUN_ID` unchanged to resume; set `RESUME=True`. A fresh run refuses to
overwrite an existing run directory. Use a new run ID for a different experiment.
Mount Drive to preserve artifacts across Colab disconnections; the dataset cache
may remain on the faster ephemeral local disk.


In [ ]:
SMOKE_TEST = True             # False explicitly starts the full 100-epoch schedule.
USE_GOOGLE_DRIVE = False      # True mounts Drive and saves persistent artifacts there.
RESUME = False               # True resumes this mode / RUN_ID from last_checkpoint.pt.
RUN_ID = "seed42"
SPLIT_SEED = 42
SEED = 42

from pathlib import Path
IN_COLAB = Path("/content").exists()
DATA_ROOT = Path("/content/data") if IN_COLAB else Path.cwd() / "data"
LOCAL_ARTIFACT_ROOT = Path("/content/clean_baseline_artifacts") if IN_COLAB else Path.cwd()
DRIVE_ARTIFACT_ROOT = Path("/content/drive/MyDrive/corruption-robustness-research")

SETTINGS = dict(
    full_epochs=100, smoke_epochs=2,
    smoke_train_per_class=2, smoke_val_per_class=1,
    batch_size=128, learning_rate=0.1, momentum=0.9, weight_decay=5e-4,
    eta_min=0.0, num_workers=2, use_amp=True, deterministic=True, cpu_threads=4, device="auto",
    download=True,
    mean=(0.5071, 0.4867, 0.4408), std=(0.2675, 0.2565, 0.2761),
)


In [ ]:
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("Drive mounting is available in Colab. Set USE_GOOGLE_DRIVE=False locally.") from exc
    drive.mount("/content/drive")
    ARTIFACT_ROOT = DRIVE_ARTIFACT_ROOT
else:
    ARTIFACT_ROOT = LOCAL_ARTIFACT_ROOT
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
import shutil
print("Artifacts:", ARTIFACT_ROOT)
print(f"Free space: {shutil.disk_usage(ARTIFACT_ROOT).free / 2**30:.1f} GiB")
if not USE_GOOGLE_DRIVE and IN_COLAB:
    print("Local Colab storage is temporary. Enable Drive for a persistent full run.")


## Reusable implementation

The following cells contain the maintained Python modules. Run them unchanged.
They write to a temporary package, without overwriting your repository files.
For development, edit `src/*.py` and run `python tools/build_notebook.py`; this
regenerates the notebook from the same sources to avoid diverging implementations.

The CIFAR stem uses a 3×3 convolution at stride 1 with no max pooling. This retains
spatial detail in 32×32 inputs. Standard ResNet18 residual blocks and stage strides
are preserved; the 100-output classifier and all other parameters start from
scratch (`weights=None`).

Training uses only RandomCrop(32, padding=4), RandomHorizontalFlip, ToTensor, and
fixed conventional CIFAR-100 normalization. Validation/test use only ToTensor
and the same normalization. The quoted constants are fixed, rounded reference
statistics; this experiment never estimates preprocessing values from held-out
images. They are not statistics recomputed on the 45,000-image subset.


In [ ]:
import sys
import tempfile
RUNTIME_ROOT = Path(tempfile.mkdtemp(prefix="clean_baseline_"))
RUNTIME_PACKAGE = str(RUNTIME_ROOT / "src")
Path(RUNTIME_PACKAGE).mkdir()
RUNTIME_ASSETS = str(RUNTIME_ROOT / "third_party")
Path(RUNTIME_ASSETS).mkdir()
sys.path.insert(0, str(RUNTIME_ROOT))
# Running all cells again should import the newly materialized sources.
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
print("Runtime package:", RUNTIME_PACKAGE)


In [ ]:
%%writefile $RUNTIME_PACKAGE/__init__.py
"""Clean CIFAR-100 baseline; no corruption-benchmark training or evaluation."""


In [ ]:
%%writefile $RUNTIME_PACKAGE/lofo.py
"""Protocol registry and auditable exposure bins; no benchmark data access."""

FAMILIES = {
    "noise": ("gaussian_noise", "shot_noise", "impulse_noise"),
    "blur": ("defocus_blur", "glass_blur", "motion_blur", "zoom_blur"),
    "weather": ("snow", "frost", "fog", "brightness"),
    "digital": ("contrast", "elastic_transform", "pixelate", "jpeg_compression"),
}
REPRESENTATIVES = {"noise": "gaussian_noise", "blur": "defocus_blur",
                   "weather": "fog", "digital": "jpeg_compression"}
PLANNED_SEEDS = (42, 123, 2026)
# Canonical JSON fingerprint of the actual Part 1 split, not a newly generated split.
ORIGINAL_SPLIT_SHA256 = "ab04334a218db1266d508e8c9adaaf91d45d81e0f4410739243ef4991fdc3ee9"
CORRUPTION_NAMES = ("clean", *REPRESENTATIVES.values())


def allowed_for(held_out_family):
    if held_out_family not in FAMILIES:
        raise ValueError(f"Unknown held-out family: {held_out_family!r}.")
    return tuple(name for family, name in REPRESENTATIVES.items() if family != held_out_family)


def exposure_bins():
    return {"clean": 0, **{f"{name}:severity{level}": 0
                            for name in CORRUPTION_NAMES[1:] for level in range(1, 6)}}


def validate_exposures(config, counts, expected_count):
    if set(counts) != set(exposure_bins()) or sum(counts.values()) != expected_count:
        raise ValueError("Corruption audit does not account for every consumed training image.")
    if any(not isinstance(value, int) or value < 0 for value in counts.values()):
        raise ValueError("Corruption counts must be nonnegative integers.")
    for name in CORRUPTION_NAMES[1:]:
        if name not in config.allowed_corruptions and any(
                counts[f"{name}:severity{s}"] for s in range(1, 6)):
            raise ValueError(f"Forbidden corruption exposure detected: {name}.")


def protocol_metadata(config):
    classification = {}
    for family, names in FAMILIES.items():
        for name in names:
            category = ("entirely_held_out_family" if family == config.held_out_family
                        else "exact_seen" if name in config.allowed_corruptions
                        else "unseen_within_represented_family")
            classification[name] = {"family": family, "exposure_category": category}
    return {
        "experiment_name": config.experiment_name, "held_out_family": config.held_out_family,
        "allowed_corruptions": list(config.allowed_corruptions), "families": FAMILIES,
        "representatives": REPRESENTATIVES, "corruption_probability": config.corruption_probability,
        "corruption_selection": config.corruption_selection,
        "severity_selection": config.severity_selection,
        "allowed_severities": list(config.allowed_severities),
        "sampling_unit": "one independent clean/corruption draw per consumed training image per epoch",
        "audit_unit": "main-process counts of consumed batches; excludes unconsumed worker prefetch",
        "corruption_implementation": config.corruption_implementation,
        "split_sha256": config.split_sha256, "initialization": "from_scratch_weights_None",
        "evaluation_categories": classification,
        "benchmark_data_used": False, "validation": "clean_only",
    }


In [ ]:
%%writefile $RUNTIME_PACKAGE/config.py
"""One serializable configuration for both notebook and command-line execution."""

from dataclasses import asdict, dataclass
from pathlib import Path
import json
from .lofo import ORIGINAL_SPLIT_SHA256


@dataclass(frozen=True)
class Config:
    smoke_test: bool = True
    seed: int = 42
    split_seed: int = 42  # Keep fixed across seeds and future experimental conditions.
    full_epochs: int = 100
    smoke_epochs: int = 2
    smoke_train_per_class: int = 2
    smoke_val_per_class: int = 1
    batch_size: int = 128
    learning_rate: float = 0.1
    momentum: float = 0.9
    weight_decay: float = 5e-4
    num_workers: int = 2
    use_amp: bool = True
    deterministic: bool = True
    cpu_threads: int = 4
    data_root: str = "data"
    artifact_root: str = "."
    run_id: str = "seed42"
    download: bool = True
    device: str = "auto"  # auto selects CUDA, then Apple MPS, then CPU.
    # Fixed protocol, serialized explicitly for future comparisons.
    dataset: str = "CIFAR100"
    num_classes: int = 100
    train_per_class: int = 450
    val_per_class: int = 50
    architecture: str = "cifar_resnet18_scratch"
    mean: tuple = (0.5071, 0.4867, 0.4408)
    std: tuple = (0.2675, 0.2565, 0.2761)
    optimizer: str = "SGD"
    scheduler: str = "CosineAnnealingLR"
    eta_min: float = 0.0
    loss: str = "CrossEntropyLoss"
    selection: str = "highest_validation_accuracy_first_tie"

    @property
    def epochs(self):
        return self.smoke_epochs if self.smoke_test else self.full_epochs

    @property
    def mode(self):
        return "smoke_test" if self.smoke_test else "clean_baseline"

    def to_dict(self):
        # JSON round-trip normalizes tuples and other config representations.
        return json.loads(json.dumps(asdict(self)))

    def validate(self):
        reference = Config()
        for key in ("dataset", "num_classes", "train_per_class", "val_per_class",
                    "architecture", "optimizer", "scheduler", "loss", "selection"):
            if getattr(self, key) != getattr(reference, key):
                raise ValueError(f"{key} is fixed by the clean-baseline protocol.")
        if self.epochs < 1 or self.batch_size < 2 or self.num_workers < 0:
            raise ValueError("Use positive epochs, batch_size >= 2 and num_workers >= 0.")
        if self.cpu_threads < 1 or self.learning_rate <= 0 or self.weight_decay < 0:
            raise ValueError("Invalid CPU thread count, learning rate or weight decay.")
        if not 0 <= self.momentum < 1 or not 0 <= self.eta_min <= self.learning_rate:
            raise ValueError("Invalid momentum or cosine minimum learning rate.")
        if not 1 <= self.smoke_train_per_class <= 450 or not 1 <= self.smoke_val_per_class <= 50:
            raise ValueError("Smoke subsets must fit within the fixed split.")
        if len(self.mean) != 3 or len(self.std) != 3 or any(s <= 0 for s in self.std):
            raise ValueError("Provide three channel means and positive standard deviations.")
        if not self.run_id or Path(self.run_id).name != self.run_id or self.run_id in (".", ".."):
            raise ValueError("run_id must be a single directory name.")
        if self.device not in ("auto", "cuda", "mps", "cpu"):
            raise ValueError("device must be auto, cuda, mps, or cpu.")
        if not 0 <= self.seed < 2**32 or not 0 <= self.split_seed < 2**32:
            raise ValueError("Seeds must be integers in [0, 2**32).")


@dataclass(frozen=True)
class LOFOConfig(Config):
    experiment_name: str = "lofo_noise"
    held_out_family: str = "noise"
    allowed_corruptions: tuple = ("defocus_blur", "fog", "jpeg_compression")
    corruption_probability: float = 0.5
    corruption_selection: str = "uniform"
    severity_selection: str = "uniform"
    allowed_severities: tuple = (1, 2, 3, 4, 5)
    split_file: str = "splits/cifar100_seed42.json"
    split_sha256: str = ORIGINAL_SPLIT_SHA256
    corruption_implementation: str = "hendrycks_cifar_c_8190fe3_v1"

    @property
    def mode(self):
        return "smoke_test" if self.smoke_test else self.experiment_name

    def validate(self):
        from .lofo import FAMILIES, ORIGINAL_SPLIT_SHA256, PLANNED_SEEDS, allowed_for
        super().validate()
        expected = allowed_for(self.held_out_family)
        if set(self.allowed_corruptions) & set(FAMILIES[self.held_out_family]):
            raise ValueError("Allowed corruptions include the entirely held-out family.")
        if self.experiment_name != f"lofo_{self.held_out_family}":
            raise ValueError("Experiment name must match its held-out family.")
        if tuple(self.allowed_corruptions) != expected:
            raise ValueError(f"Use exactly the predefined representatives in order: {expected}.")
        if not 0 <= self.corruption_probability <= 1:
            raise ValueError("corruption_probability must be in [0,1].")
        if (self.corruption_selection != "uniform" or self.severity_selection != "uniform"
                or tuple(self.allowed_severities) != (1, 2, 3, 4, 5)):
            raise ValueError("Use uniform selection over the three representatives and severities 1–5.")
        if self.seed not in PLANNED_SEEDS:
            raise ValueError(f"Planned LOFO training seeds are {PLANNED_SEEDS}.")
        if self.run_id != f"seed{self.seed}" and not self.run_id.endswith(f"_seed{self.seed}"):
            raise ValueError("LOFO run_id must be seedN or end with _seedN for this seed.")
        if (self.split_seed != 42 or self.split_sha256 != ORIGINAL_SPLIT_SHA256
                or not self.split_file):
            raise ValueError("LOFO requires the original persisted Part 1 seed-42 split and fingerprint.")
        if self.corruption_implementation != LOFOConfig().corruption_implementation:
            raise ValueError("Unknown corruption implementation/version.")
        baseline = Config()
        for key in ("full_epochs", "batch_size", "learning_rate", "momentum", "weight_decay",
                    "eta_min", "mean", "std"):
            value, reference = getattr(self, key), getattr(baseline, key)
            if key in ("mean", "std"):
                value, reference = list(value), list(reference)
            if value != reference:
                raise ValueError(f"{key} must match the predefined clean baseline for fair comparisons.")


def load_config(path):
    with open(path, encoding="utf-8") as handle:
        data = json.load(handle)
        config = (LOFOConfig if "experiment_name" in data else Config)(**data)
    config.validate()
    return config


In [ ]:
%%writefile $RUNTIME_PACKAGE/utils.py
"""Reproducibility, portable checkpoints, and atomic artifact writes."""

import os
# Must be set before CUDA context creation (including notebook diagnostics).
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import csv
import hashlib
import json
import platform
import random
import subprocess
import sys
import uuid
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torchvision


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def atomic_write(path, writer):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    try:
        writer(temporary)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def save_json(path, payload):
    atomic_write(path, lambda p: p.write_text(
        json.dumps(payload, indent=2, allow_nan=False) + "\n", encoding="utf-8"))


def save_checkpoint(path, payload):
    atomic_write(path, lambda p: torch.save(payload, p))


def load_checkpoint(path):
    # Store only tensors and primitive containers; no arbitrary object unpickling.
    return torch.load(path, map_location="cpu", weights_only=True)


def seed_everything(seed, deterministic=True, cpu_threads=4):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    torch.set_num_threads(cpu_threads)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    # Fail explicitly rather than quietly use a nondeterministic operation.
    torch.use_deterministic_algorithms(deterministic)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def capture_rng():
    numpy_state = np.random.get_state()
    return {
        "python": random.getstate(),
        "numpy": [numpy_state[0], numpy_state[1].tolist(), *numpy_state[2:]],
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
        "mps": torch.mps.get_rng_state() if torch.backends.mps.is_available() else None,
    }


def restore_rng(state):
    random.setstate(state["python"])
    numpy_state = state["numpy"]
    np.random.set_state((numpy_state[0], np.asarray(numpy_state[1], dtype=np.uint32),
                         *numpy_state[2:]))
    torch.set_rng_state(state["torch"])
    if state.get("mps") is not None and torch.backends.mps.is_available():
        torch.mps.set_rng_state(state["mps"])
    if state["cuda"] and torch.cuda.is_available():
        if len(state["cuda"]) == torch.cuda.device_count():
            torch.cuda.set_rng_state_all(state["cuda"])
        else:
            warnings.warn("CUDA device count changed; exact replay is not guaranteed.")


def select_device(preference="auto"):
    available = {"cuda": torch.cuda.is_available(), "mps": torch.backends.mps.is_available(), "cpu": True}
    if preference == "auto":
        return torch.device(next(name for name in ("cuda", "mps", "cpu") if available[name]))
    if preference not in available or not available[preference]:
        raise RuntimeError(f"Requested device {preference!r} is unavailable. "
                           "Use --device cpu for smoke checks, or enable the requested accelerator.")
    return torch.device(preference)


def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def model_digest(model):
    digest = hashlib.sha256()
    for name, tensor in model.state_dict().items():
        digest.update(name.encode())
        digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def environment_info(device):
    try:
        commit = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL, text=True).strip()
        dirty = bool(subprocess.check_output(
            ["git", "status", "--porcelain"], stderr=subprocess.DEVNULL, text=True).strip())
    except (OSError, subprocess.CalledProcessError):
        commit, dirty = None, None
    gpu = torch.cuda.get_device_properties(device) if device.type == "cuda" else None
    return {
        "python": sys.version, "platform": platform.platform(),
        "torch": str(torch.__version__), "torchvision": str(torchvision.__version__),
        "numpy": str(np.__version__), "cuda": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(), "device": str(device),
        "gpu_model": gpu.name if gpu else "Apple Metal (MPS)" if device.type == "mps" else None,
        "mps_available": torch.backends.mps.is_available(),
        "gpu_memory_gib": gpu.total_memory / 2**30 if gpu else None,
        "git_commit": commit, "git_dirty": dirty,
        "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
        "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
        "cpu_threads": torch.get_num_threads(),
    }


def source_digest():
    sources = sorted(Path(__file__).parent.glob("*.py"))
    digest = hashlib.sha256()
    for path in sources:
        digest.update(path.name.encode())
        digest.update(path.read_bytes())
    return digest.hexdigest()


def write_metrics(output_dir, history):
    output_dir = Path(output_dir)
    save_json(output_dir / "metrics.json", history)
    if history:
        def writer(path):
            with path.open("w", newline="", encoding="utf-8") as handle:
                csv_writer = csv.DictWriter(handle, fieldnames=list(history[0]))
                csv_writer.writeheader()
                csv_writer.writerows(history)
        atomic_write(output_dir / "metrics.csv", writer)


In [ ]:
%%writefile $RUNTIME_PACKAGE/model.py
"""Identical scratch architecture for all future experimental conditions."""

from torch import nn
from torchvision.models import resnet18


def build_model():
    model = resnet18(weights=None, num_classes=100)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    # Match torchvision's Kaiming initialization for the replacement convolution.
    nn.init.kaiming_normal_(model.conv1.weight, mode="fan_out", nonlinearity="relu")
    model.maxpool = nn.Identity()
    return model


In [ ]:
%%writefile $RUNTIME_PACKAGE/corruptions.py
"""Training-only CIFAR corruption operators adapted from Hendrycks & Dietterich.

Apache-2.0 adaptations of make_cifar_c.py at commit
8190fe329f5f072a06e3a2aea02bb5dda69aed9f; see third_party/NOTICE.md and license.
MODIFIED: explicit RNG injection, checked PIL interface, NumPy 2 dtype alias,
uint8 outputs, worker-safe exposure metadata. No benchmark dataset access.
"""
from io import BytesIO
from functools import lru_cache
import random
import numpy as np
from PIL import Image as PILImage, features, __version__ as pillow_version
from torchvision import transforms
from .lofo import CORRUPTION_NAMES
try:
    import cv2
except ImportError:
    cv2 = None

UPSTREAM_COMMIT = "8190fe329f5f072a06e3a2aea02bb5dda69aed9f"


@lru_cache(maxsize=8)
def _disk(radius, alias_blur=0.1, dtype=np.float32):
    if radius <= 8:
        L = np.arange(-8, 8 + 1)
        ksize = (3, 3)
    else:
        L = np.arange(-radius, radius + 1)
        ksize = (5, 5)
    X, Y = np.meshgrid(L, L)
    aliased_disk = np.array((X ** 2 + Y ** 2) <= radius ** 2, dtype=dtype)
    aliased_disk /= np.sum(aliased_disk)

    # supersample disk to antialias
    return cv2.GaussianBlur(aliased_disk, ksize=ksize, sigmaX=alias_blur)


def _plasma_fractal(mapsize=32, wibbledecay=3, rng=None):
    """
    Generate a heightmap using diamond-square algorithm.
    Return square 2d array, side length 'mapsize', of floats in range 0-255.
    'mapsize' must be a power of two.
    """
    assert (mapsize & (mapsize - 1) == 0)
    maparray = np.empty((mapsize, mapsize), dtype=np.float64)
    maparray[0, 0] = 0
    stepsize = mapsize
    wibble = 100

    def wibbledmean(array):
        return array / 4 + wibble * rng.uniform(-wibble, wibble, array.shape)

    def fillsquares():
        """For each square of points stepsize apart,
           calculate middle value as mean of points + wibble"""
        cornerref = maparray[0:mapsize:stepsize, 0:mapsize:stepsize]
        squareaccum = cornerref + np.roll(cornerref, shift=-1, axis=0)
        squareaccum += np.roll(squareaccum, shift=-1, axis=1)
        maparray[stepsize // 2:mapsize:stepsize,
        stepsize // 2:mapsize:stepsize] = wibbledmean(squareaccum)

    def filldiamonds():
        """For each diamond of points stepsize apart,
           calculate middle value as mean of points + wibble"""
        mapsize = maparray.shape[0]
        drgrid = maparray[stepsize // 2:mapsize:stepsize, stepsize // 2:mapsize:stepsize]
        ulgrid = maparray[0:mapsize:stepsize, 0:mapsize:stepsize]
        ldrsum = drgrid + np.roll(drgrid, 1, axis=0)
        lulsum = ulgrid + np.roll(ulgrid, -1, axis=1)
        ltsum = ldrsum + lulsum
        maparray[0:mapsize:stepsize, stepsize // 2:mapsize:stepsize] = wibbledmean(ltsum)
        tdrsum = drgrid + np.roll(drgrid, 1, axis=1)
        tulsum = ulgrid + np.roll(ulgrid, -1, axis=0)
        ttsum = tdrsum + tulsum
        maparray[stepsize // 2:mapsize:stepsize, 0:mapsize:stepsize] = wibbledmean(ttsum)

    while stepsize >= 2:
        fillsquares()
        filldiamonds()
        stepsize //= 2
        wibble /= wibbledecay

    maparray -= maparray.min()
    return maparray / maparray.max()


def _gaussian_noise(x, severity=1, rng=None):
    c = [0.04, 0.06, .08, .09, .10][severity - 1]

    x = np.array(x) / 255.
    return np.clip(x + rng.normal(size=x.shape, scale=c), 0, 1) * 255


def _defocus_blur(x, severity=1, rng=None):
    c = [(0.3, 0.4), (0.4, 0.5), (0.5, 0.6), (1, 0.2), (1.5, 0.1)][severity - 1]

    x = np.array(x) / 255.
    kernel = _disk(radius=c[0], alias_blur=c[1])

    channels = []
    for d in range(3):
        channels.append(cv2.filter2D(x[:, :, d], -1, kernel))
    channels = np.array(channels).transpose((1, 2, 0))  # 3x32x32 -> 32x32x3

    return np.clip(channels, 0, 1) * 255


def _fog(x, severity=1, rng=None):
    c = [(.2,3), (.5,3), (0.75,2.5), (1,2), (1.5,1.75)][severity - 1]

    x = np.array(x) / 255.
    max_val = x.max()
    x += c[0] * _plasma_fractal(wibbledecay=c[1], rng=rng)[:32, :32][..., np.newaxis]
    return np.clip(x * max_val / (max_val + c[0]), 0, 1) * 255


def _jpeg_compression(x, severity=1, rng=None):
    c = [80, 65, 58, 50, 40][severity - 1]

    output = BytesIO()
    x.save(output, 'JPEG', quality=c)
    x = PILImage.open(output)

    return x


OPERATORS = {"gaussian_noise": _gaussian_noise, "defocus_blur": _defocus_blur,
             "fog": _fog, "jpeg_compression": _jpeg_compression}


def require_opencv():
    if cv2 is None:
        raise RuntimeError("Defocus blur requires OpenCV. Install opencv-python-headless "
                           "(requirements-lofo.txt), or use Colab's compatible preinstalled cv2.")
    cv2.setNumThreads(1)  # Avoid worker-by-OpenCV thread oversubscription.


def implementation_metadata():
    return {"source_commit": UPSTREAM_COMMIT, "operator_set": list(OPERATORS),
            "opencv": None if cv2 is None else cv2.__version__,
            "pillow": pillow_version, "jpeg_codec": features.version_codec("jpg"),
            "numpy": np.__version__, "serialization": "clip_then_truncate_uint8_RGB",
            "identity_claim": "CIFAR-reference operators; not released benchmark image identity"}


def apply_corruption(image, name, severity, rng=None):
    if name not in OPERATORS:
        raise ValueError(f"Unsupported training corruption: {name!r}.")
    if isinstance(severity, bool) or not isinstance(severity, (int, np.integer)) or not 1 <= severity <= 5:
        raise ValueError("Severity must be an integer from 1 through 5.")
    if not isinstance(image, PILImage.Image) or image.mode != "RGB" or image.size != (32, 32):
        raise ValueError("Corruptions require a 32×32 RGB PIL image before tensor conversion.")
    if name == "defocus_blur":
        require_opencv()
    rng = np.random if rng is None else rng
    result = OPERATORS[name](image, severity, rng=rng)
    # Same quantization used by the original benchmark's np.uint8 serialization.
    pixels = np.asarray(result)
    if pixels.shape != (32, 32, 3) or not np.isfinite(pixels).all():
        raise ValueError(f"{name} produced invalid image pixels.")
    return PILImage.fromarray(np.clip(pixels, 0, 255).astype(np.uint8))


def gaussian_noise(image, severity=1, rng=None):
    return apply_corruption(image, "gaussian_noise", severity, rng)


def defocus_blur(image, severity=1, rng=None):
    return apply_corruption(image, "defocus_blur", severity, rng)


def fog(image, severity=1, rng=None):
    return apply_corruption(image, "fog", severity, rng)


def jpeg_compression(image, severity=1, rng=None):
    return apply_corruption(image, "jpeg_compression", severity, rng)


class LOFOTrainingTransform:
    """One spatial augmentation, then zero/one corruption, then tensor+normalize.

    NumPy's worker-local RNG handles sampling. Python seeds a fresh operator RNG
    for each corrupted image; operator draw counts cannot alter future selections.
    Torch handles crop/flip, independently of both corruption streams.
    Main-process RNG and loader worker seeds are already checkpointed by Part 1.
    """

    def __init__(self, config):
        config.validate()
        self.allowed = tuple(config.allowed_corruptions)
        self.probability = config.corruption_probability
        self.severities = tuple(config.allowed_severities)
        if "defocus_blur" in self.allowed:
            require_opencv()
        self.spatial = transforms.Compose([
            transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()])
        self.tensor = transforms.Compose([
            transforms.ToTensor(), transforms.Normalize(config.mean, config.std)])

    def sample(self, rng=None):
        rng = np.random if rng is None else rng
        if rng.random() >= self.probability:
            return "clean", 0
        return str(rng.choice(self.allowed)), int(rng.choice(self.severities))

    def __call__(self, image):
        image = self.spatial(image)
        name, severity = self.sample()
        if name != "clean":
            operator_rng = np.random.RandomState(random.getrandbits(32))
            image = apply_corruption(image, name, severity, rng=operator_rng)
        return self.tensor(image), CORRUPTION_NAMES.index(name), severity


In [ ]:
%%writefile $RUNTIME_PACKAGE/dataset.py
"""Official training-only split, independent transforms, and seeded loaders."""

import hashlib
import json
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

from .utils import save_json, seed_worker
from .config import LOFOConfig


def split_fingerprint(split):
    encoded = json.dumps(split, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(encoded).hexdigest()


def validate_split(split, targets, split_seed):
    targets = np.asarray(targets, dtype=np.int64)
    if targets.shape != (50000,) or not np.array_equal(np.bincount(targets, minlength=100),
                                                      np.full(100, 500)):
        raise ValueError("Expected official CIFAR-100 training targets: 500 per class.")
    label_hash = hashlib.sha256(targets.astype("<i8").tobytes()).hexdigest()
    if (split["split_seed"] != split_seed or split["targets_sha256"] != label_hash
            or split["dataset"] != "CIFAR100_official_train" or split["version"] != 1):
        raise ValueError("Saved split does not match the dataset or split seed.")
    train = np.asarray(split["train_indices"])
    val = np.asarray(split["val_indices"])
    if (train.shape != (45000,) or val.shape != (5000,)
            or not np.issubdtype(train.dtype, np.integer)
            or not np.issubdtype(val.dtype, np.integer)):
        raise ValueError("Saved split must contain 45,000/5,000 integer indices.")
    all_indices = np.concatenate([train, val])
    if not np.array_equal(np.sort(all_indices), np.arange(50000)):
        raise ValueError("Split contains overlaps, omissions, or invalid indices.")
    for indices, expected in ((train, 450), (val, 50)):
        if not np.array_equal(np.bincount(targets[indices], minlength=100), np.full(100, expected)):
            raise ValueError("Split is not exactly stratified (450 train / 50 val per class).")


def get_or_create_split(targets, path, split_seed=42):
    path = Path(path)
    targets = np.asarray(targets, dtype=np.int64)
    if path.exists():
        split = json.loads(path.read_text(encoding="utf-8"))
    else:
        if targets.shape != (50000,) or not np.array_equal(
                np.bincount(targets, minlength=100), np.full(100, 500)):
            raise ValueError("Only the 50,000 official training images may define the split.")
        rng = np.random.default_rng(split_seed)
        train, val = [], []
        for label in range(100):
            indices = rng.permutation(np.flatnonzero(targets == label))
            val.extend(indices[:50].tolist())
            train.extend(indices[50:].tolist())
        split = {
            "version": 1, "dataset": "CIFAR100_official_train", "split_seed": split_seed,
            "targets_sha256": hashlib.sha256(targets.astype("<i8").tobytes()).hexdigest(),
            "train_indices": sorted(train), "val_indices": sorted(val),
        }
        validate_split(split, targets, split_seed)
        save_json(path, split)
    validate_split(split, targets, split_seed)
    return split


def balanced_subset(indices, targets, per_class, seed):
    indices = np.asarray(indices)
    targets = np.asarray(targets)
    rng = np.random.default_rng(seed)
    chosen = []
    for label in range(100):
        candidates = indices[targets[indices] == label]
        if len(candidates) < per_class:
            raise ValueError(f"Insufficient examples for class {label}.")
        chosen.extend(rng.permutation(candidates)[:per_class].tolist())
    return sorted(chosen)


class TransformSubset(Dataset):
    """Share raw images, but apply distinct training and validation transforms."""

    def __init__(self, raw, indices, transform):
        self.raw, self.indices, self.transform = raw, list(indices), transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        image, label = self.raw[self.indices[index]]
        transformed = self.transform(image)
        if isinstance(transformed, tuple):
            tensor, corruption_code, severity = transformed
            return tensor, label, corruption_code, severity
        return transformed, label


def make_transforms(config):
    normalize = transforms.Normalize(config.mean, config.std)
    train = transforms.Compose([
        transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
        transforms.ToTensor(), normalize,
    ])
    evaluation = transforms.Compose([transforms.ToTensor(), normalize])
    return train, evaluation


def make_loader(dataset, config, device, generator, shuffle=False):
    return DataLoader(
        dataset, batch_size=config.batch_size, shuffle=shuffle,
        num_workers=config.num_workers, pin_memory=device.type == "cuda",
        worker_init_fn=seed_worker, generator=generator,
        # Recreate workers each epoch so restored generator state reproduces worker seeds.
        persistent_workers=False, drop_last=False,
    )


def prepare_data(config, device):
    config.validate()
    lofo = isinstance(config, LOFOConfig)
    if lofo:
        # Fail before downloading/loading any data if the original split is absent.
        split_path = Path(config.split_file)
        if not split_path.is_file():
            raise FileNotFoundError(f"Original Part 1 split required: {split_path}. "
                                    "Copy the persisted split or set --split-file; LOFO never regenerates it.")
        split = json.loads(split_path.read_text(encoding="utf-8"))
        if split_fingerprint(split) != config.split_sha256:
            raise ValueError("LOFO split fingerprint differs from the original Part 1 split.")
    try:
        raw = datasets.CIFAR100(config.data_root, train=True, download=config.download)
    except (OSError, RuntimeError) as exc:
        raise RuntimeError(
            f"Could not load official CIFAR-100 from {config.data_root}. "
            "Enable internet for the first download (~170 MB archive), check free disk "
            "space, or supply an existing torchvision CIFAR-100 cache."
        ) from exc
    if lofo:
        validate_split(split, raw.targets, config.split_seed)
    else:
        split_path = Path(config.artifact_root) / "splits" / f"cifar100_seed{config.split_seed}.json"
        split = get_or_create_split(raw.targets, split_path, config.split_seed)
    train_indices, val_indices = split["train_indices"], split["val_indices"]
    if config.smoke_test:
        train_indices = balanced_subset(train_indices, raw.targets,
                                        config.smoke_train_per_class, config.split_seed + 1)
        val_indices = balanced_subset(val_indices, raw.targets,
                                      config.smoke_val_per_class, config.split_seed + 2)
    train_transform, eval_transform = make_transforms(config)
    if lofo:
        from .corruptions import LOFOTrainingTransform
        train_transform = LOFOTrainingTransform(config)
    generators = {name: torch.Generator().manual_seed(config.seed + offset)
                  for name, offset in (("train", 0), ("val", 1))}
    train_loader = make_loader(TransformSubset(raw, train_indices, train_transform),
                               config, device, generators["train"], shuffle=True)
    val_loader = make_loader(TransformSubset(raw, val_indices, eval_transform),
                             config, device, generators["val"])
    manifest = {
        "split_path": str(split_path), "split_sha256": split_fingerprint(split),
        "train_indices": train_indices, "val_indices": val_indices,
        "train_count": len(train_indices), "val_count": len(val_indices),
        "classes": raw.classes,
    }
    print(f"{config.mode}: {len(train_indices):,} training / {len(val_indices):,} validation; "
          "official test set is not loaded.")
    return train_loader, val_loader, generators, manifest, split


In [ ]:
%%writefile $RUNTIME_PACKAGE/train.py
"""Clean baseline and LOFO training; official test images are never loaded here."""

import argparse
import json
import math
import shutil
import time
import warnings
from pathlib import Path
from dataclasses import replace
from contextlib import nullcontext

import torch
from torch import nn

from .config import LOFOConfig, load_config
from .lofo import CORRUPTION_NAMES, exposure_bins, protocol_metadata, validate_exposures
from .dataset import prepare_data
from .model import build_model
from .utils import (capture_rng, environment_info, load_checkpoint, restore_rng,
                    save_checkpoint, save_json, seed_everything, select_device,
                    source_digest, utc_now, write_metrics, synchronize, model_digest)


def run_directory(config):
    if isinstance(config, LOFOConfig) and config.smoke_test:
        return Path(config.artifact_root) / "results" / "smoke_test" / config.experiment_name / config.run_id
    return Path(config.artifact_root) / "results" / config.mode / config.run_id


def compatible_config(saved, current):
    # Relocating Drive/cache directories does not change the experiment.
    ignored = {"data_root", "artifact_root", "download", "split_file"}
    return {k: v for k, v in saved.items() if k not in ignored} == {
        k: v for k, v in current.items() if k not in ignored}


def write_corruption_audit(output_dir, history):
    totals = exposure_bins()
    for row in history:
        for key, value in row["counts"].items():
            totals[key] += value
    save_json(Path(output_dir) / "corruption_counts.json", {
        "unit": "actually_consumed_training_images", "epochs": history, "totals": totals,
        "total_images": sum(totals.values()), "clean_images": totals["clean"],
        "corrupted_images": sum(totals.values()) - totals["clean"],
    })


def run_epoch(model, loader, criterion, device, optimizer=None, scaler=None, amp=False):
    training = optimizer is not None
    model.train(training)
    # Metal does not support float64. CPU/CUDA retain Part 1's float64 accumulation.
    accumulator_dtype = torch.float32 if device.type == "mps" else torch.float64
    total_loss = torch.zeros((), device=device, dtype=accumulator_dtype)
    total_correct = torch.zeros((), device=device, dtype=torch.int64)
    count = 0
    exposures = None
    context = torch.enable_grad if training else torch.inference_mode
    with context():
        for batch in loader:
            images, targets = batch[:2]
            if len(batch) == 4:
                if not training:
                    raise ValueError("Corruption metadata is forbidden in clean validation/test batches.")
                if exposures is None:
                    exposures = exposure_bins()
                codes, severities = batch[2:]
                for code, severity in zip(codes.tolist(), severities.tolist()):
                    if not 0 <= code < len(CORRUPTION_NAMES) or not (severity == 0 if code == 0 else 1 <= severity <= 5):
                        raise ValueError("Invalid corruption audit code/severity.")
                    name = CORRUPTION_NAMES[code]
                    exposures["clean" if code == 0 else f"{name}:severity{severity}"] += 1
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type) if amp else nullcontext():
                logits = model(images)
                loss = criterion(logits, targets)
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite loss; inspect configuration and inputs.")
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            size = targets.size(0)
            count += size
            total_loss += loss.detach().to(accumulator_dtype) * size
            total_correct += (logits.detach().argmax(1) == targets).sum()
    if not count:
        raise ValueError("An empty loader cannot be trained or evaluated.")
    result = {"loss": total_loss.item() / count,
              "accuracy_pct": 100.0 * total_correct.item() / count, "count": count}
    if exposures is not None:
        result["corruption_counts"] = exposures
    return result


def plot_history(history, output_dir, smoke_test, show=False, experiment_name="Clean CIFAR-100 baseline"):
    import matplotlib.pyplot as plt

    if not history:
        raise ValueError("No epoch metrics to plot.")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    epochs = [row["epoch"] for row in history]
    label = f"SMOKE TEST — {experiment_name}" if smoke_test else experiment_name
    specs = [
        ("accuracy", "Accuracy (%)", [("train_accuracy_pct", "Training", "#0072B2"),
                                       ("val_accuracy_pct", "Validation", "#D55E00")]),
        ("loss", "Cross-entropy loss", [("train_loss", "Training", "#0072B2"),
                                        ("val_loss", "Validation", "#D55E00")]),
        ("learning_rate", "Learning rate", [("learning_rate", "SGD", "#0072B2")]),
    ]
    with plt.rc_context({"font.size": 11, "axes.spines.top": False,
                         "axes.spines.right": False, "savefig.dpi": 300}):
        for filename, ylabel, series in specs:
            fig, ax = plt.subplots(figsize=(6.4, 4.2), layout="constrained")
            for key, legend, color in series:
                ax.plot(epochs, [row[key] for row in history], label=legend,
                        color=color, linewidth=1.8, marker="o" if len(epochs) < 10 else None)
            ax.set(xlabel="Epoch", ylabel=ylabel, title=label)
            if filename == "accuracy":
                ax.set_ylim(0, 100)
            else:
                maximum = max(row[key] for key, _, _ in series for row in history)
                ax.set_ylim(0, maximum * 1.08 if maximum > 0 else 1)
            from matplotlib.ticker import MaxNLocator
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
            ax.grid(alpha=0.22)
            ax.legend(frameon=False)
            fig.savefig(output_dir / f"{filename}.png", dpi=300)
            if show:
                plt.show()
            plt.close(fig)


def run_experiment(config, resume=False, stop_after_epoch=None):
    """Checkpoint at epoch boundaries; a partial epoch is replayed after interruption.

    stop_after_epoch allows a deliberate pause without changing the planned cosine
    schedule. It is also used by resume regression tests. No test-set evaluation.
    """
    config.validate()
    lofo = isinstance(config, LOFOConfig)
    if stop_after_epoch is not None and not 1 <= stop_after_epoch <= config.epochs:
        raise ValueError("stop_after_epoch must be within the planned training schedule.")
    output_dir = run_directory(config)
    checkpoint_dir = output_dir / "checkpoints"
    last_path, best_path = (checkpoint_dir / "last_checkpoint.pt",
                            checkpoint_dir / "best_model.pt")
    if resume and not last_path.exists():
        raise FileNotFoundError(f"Cannot resume: {last_path} does not exist.")
    if not resume and output_dir.exists() and any(output_dir.iterdir()):
        raise FileExistsError(f"{output_dir} already contains artifacts. Use resume=True "
                              "or choose a new run_id to preserve the previous run.")

    seed_everything(config.seed, config.deterministic, config.cpu_threads)
    device = select_device() if config.device == "auto" else select_device(config.device)
    amp = config.use_amp and device.type == "cuda" and torch.cuda.get_device_capability(device)[0] >= 7
    environment = environment_info(device)
    if lofo:
        from .corruptions import implementation_metadata
        environment["corruptions"] = implementation_metadata()
    print(json.dumps({**environment, "effective_amp": amp}, indent=2))
    if device.type == "cpu" and not config.smoke_test:
        warnings.warn("Full training on CPU is slow. Enable a Colab GPU before starting.")
    train_loader, val_loader, generators, data_manifest, split = prepare_data(config, device)
    model = build_model().to(device)
    initial_model_sha256 = model_digest(model)
    optimizer = torch.optim.SGD(model.parameters(), lr=config.learning_rate,
                                momentum=config.momentum, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config.epochs, eta_min=config.eta_min)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    criterion = nn.CrossEntropyLoss()
    history, best_accuracy, best_epoch, start_epoch, elapsed = [], -math.inf, 0, 1, 0.0
    corruption_history = []
    digest = source_digest()
    session = {"started_utc": utc_now(), "environment": environment, "effective_amp": amp}

    if resume:
        checkpoint = load_checkpoint(last_path)
        if not compatible_config(checkpoint["config"], config.to_dict()):
            raise ValueError("Resume configuration differs from the saved experiment; "
                             "keep the original schedule/settings or use a new run_id.")
        if checkpoint["source_sha256"] != digest:
            raise ValueError("Implementation changed since this checkpoint; restore its "
                             "saved source snapshot before resuming.")
        if checkpoint["data_manifest"] != data_manifest:
            # Ignore the location of the split file when Drive/cache is relocated.
            old = {k: v for k, v in checkpoint["data_manifest"].items() if k != "split_path"}
            new = {k: v for k, v in data_manifest.items() if k != "split_path"}
            if old != new:
                raise ValueError("Resume data indices/classes differ from the saved run.")
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        if amp and checkpoint["scaler_state_dict"]:
            scaler.load_state_dict(checkpoint["scaler_state_dict"])
        if checkpoint["environment"] != environment or checkpoint["effective_amp"] != amp:
            warnings.warn("Runtime or precision changed. Loading is portable, but bitwise "
                          "replay is only expected on the same hardware/software stack.")
        history = checkpoint["history"]
        if lofo:
            corruption_history = checkpoint["corruption_history"]
            if len(corruption_history) != len(history):
                raise ValueError("Resume corruption history is incomplete.")
            for row in corruption_history:
                validate_exposures(config, row["counts"], data_manifest["train_count"])
        best_accuracy, best_epoch = checkpoint["best_validation_accuracy_pct"], checkpoint["best_epoch"]
        start_epoch, elapsed = checkpoint["epoch"] + 1, checkpoint["total_training_seconds"]
        # Recover a crash between committing 'last' and publishing a new 'best'.
        best = load_checkpoint(best_path) if best_path.exists() else None
        if best is None or best["epoch"] != best_epoch:
            if checkpoint["epoch"] == best_epoch:
                save_checkpoint(best_path, checkpoint)
            else:
                raise RuntimeError("Best checkpoint is missing/inconsistent. Restore it "
                                   "from backup; the last model cannot replace an earlier best.")
        restore_rng(checkpoint["rng_state"])
        for name, generator in generators.items():
            generator.set_state(checkpoint["loader_rng_state"][name])
        sessions = json.loads((output_dir / "sessions.json").read_text())
    else:
        output_dir.mkdir(parents=True, exist_ok=True)
        save_json(output_dir / "config.json", config.to_dict())
        save_json(output_dir / "split.json", split)
        save_json(output_dir / "data_manifest.json", data_manifest)
        save_json(output_dir / "environment.json", environment)
        (output_dir / "model.txt").write_text(str(model) + "\n", encoding="utf-8")
        shutil.copytree(Path(__file__).parent, output_dir / "source",
                        ignore=shutil.ignore_patterns("__pycache__"))
        notices = Path(__file__).parent.parent / "third_party"
        if notices.exists():
            shutil.copytree(notices, output_dir / "third_party",
                            ignore=shutil.ignore_patterns("__pycache__"))
        save_json(output_dir / "protocol.json", {
            "mode": config.mode, "pipeline_check_only": config.smoke_test,
            "source_sha256": digest, "scheduler_T_max": config.epochs,
            "effective_amp": amp, "parameter_count": sum(p.numel() for p in model.parameters()),
            "initial_model_sha256": initial_model_sha256,
            "selection": config.selection, "test_set_used_for_selection": False,
            "train_transforms": ["RandomCrop(32,padding=4)", "RandomHorizontalFlip(p=0.5)"]
                                + (["LOFO:zero_or_one_corruption_before_normalization"] if lofo else [])
                                + ["ToTensor", "Normalize(config.mean,config.std)"],
            "eval_transforms": ["ToTensor", "Normalize(config.mean,config.std)"],
            **(protocol_metadata(config) if lofo else {}),
        })
        sessions = []
    sessions.append(session)
    save_json(output_dir / "sessions.json", sessions)
    with (output_dir / "training.log").open("a", encoding="utf-8") as log:
        log.write(json.dumps(session) + "\n")
    # Checkpoint history is authoritative if a disconnect interrupted metric exports.
    write_metrics(output_dir, history)
    if lofo:
        write_corruption_audit(output_dir, corruption_history)
    final_epoch = min(config.epochs, stop_after_epoch or config.epochs)
    for epoch in range(start_epoch, final_epoch + 1):
        epoch_start = time.perf_counter()
        learning_rate = optimizer.param_groups[0]["lr"]  # LR actually used this epoch.
        synchronize(device)
        train_start = time.perf_counter()
        train_metrics = run_epoch(model, train_loader, criterion, device, optimizer, scaler, amp)
        synchronize(device)
        train_seconds = time.perf_counter() - train_start
        val_metrics = run_epoch(model, val_loader, criterion, device, amp=amp)
        synchronize(device)
        epoch_seconds = time.perf_counter() - epoch_start
        elapsed += epoch_seconds
        improved = val_metrics["accuracy_pct"] > best_accuracy  # First epoch wins ties.
        if improved:
            best_accuracy, best_epoch = val_metrics["accuracy_pct"], epoch
        scheduler.step()  # Saved state is ready for the next epoch, including on resume.
        history.append({
            "mode": config.mode, "epoch": epoch, "learning_rate": learning_rate,
            "train_loss": train_metrics["loss"], "train_accuracy_pct": train_metrics["accuracy_pct"],
            "val_loss": val_metrics["loss"], "val_accuracy_pct": val_metrics["accuracy_pct"],
            "train_seconds": train_seconds, "epoch_seconds": epoch_seconds,
            "total_training_seconds": elapsed,
        })
        if lofo:
            counts = train_metrics.get("corruption_counts")
            if counts is None:
                raise ValueError("LOFO training requires consumed-batch corruption audit metadata.")
            validate_exposures(config, counts, train_metrics["count"])
            if train_metrics["count"] != data_manifest["train_count"]:
                raise ValueError("Training did not consume every selected training image exactly once.")
            corruption_history.append({"epoch": epoch, "counts": counts})
        checkpoint = {
            "format_version": 1, "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(), "scaler_state_dict": scaler.state_dict(),
            "epoch": epoch, "best_validation_accuracy_pct": best_accuracy, "best_epoch": best_epoch,
            "config": config.to_dict(), "seed": config.seed, "source_sha256": digest,
            "environment": environment, "effective_amp": amp, "data_manifest": data_manifest,
            "rng_state": capture_rng(),
            "loader_rng_state": {name: gen.get_state() for name, gen in generators.items()},
            "history": history, "total_training_seconds": elapsed,
            **({"corruption_history": corruption_history,
                "lofo_protocol": protocol_metadata(config)} if lofo else {}),
        }
        save_checkpoint(last_path, checkpoint)
        if improved:
            save_checkpoint(best_path, checkpoint)
        write_metrics(output_dir, history)
        if lofo:
            write_corruption_audit(output_dir, corruption_history)
        message = (f"{config.experiment_name if lofo else config.mode} | epoch {epoch}/{config.epochs} | lr {learning_rate:.6f} | "
              f"train {train_metrics['accuracy_pct']:.2f}% | val {val_metrics['accuracy_pct']:.2f}% | "
              f"{epoch_seconds:.1f}s")
        print(message, flush=True)
        with (output_dir / "training.log").open("a", encoding="utf-8") as log:
            log.write(message + "\n")

    completed = len(history) == config.epochs
    save_json(output_dir / "summary.json", {
        "mode": config.mode, "scientific_result": not config.smoke_test and completed,
        **({"experiment_name": config.experiment_name} if lofo else {}),
        "completed": completed, "completed_epochs": len(history), "planned_epochs": config.epochs,
        "best_epoch": best_epoch, "best_validation_accuracy_pct": best_accuracy,
        "total_training_seconds": elapsed,
        "timing_definition": "Sum of train+validation epoch durations; excludes setup, "
                             "checkpoint/metric writes, plots, and downtime between sessions.",
        "test_evaluated": (output_dir / "clean_test.json").exists(),
    })
    if history:
        plot_history(history, output_dir / "plots", config.smoke_test,
                     experiment_name=config.experiment_name if lofo else "Clean CIFAR-100 baseline")
    return output_dir, history


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", default="configs/baseline.json")
    parser.add_argument("--resume", action="store_true")
    modes = parser.add_mutually_exclusive_group()
    modes.add_argument("--smoke-test", action="store_true")
    modes.add_argument("--full", action="store_true", help="Explicitly run the predefined 100-epoch schedule")
    parser.add_argument("--seed", type=int)
    parser.add_argument("--run-id")
    parser.add_argument("--device", choices=("auto", "cuda", "mps", "cpu"))
    parser.add_argument("--data-root")
    parser.add_argument("--artifact-root")
    parser.add_argument("--split-file", help="LOFO only: existing Part 1 split; never regenerated")
    parser.add_argument("--num-workers", type=int)
    parser.add_argument("--no-download", action="store_true")
    parser.add_argument("--stop-after-epoch", type=int, help="Pause at this epoch without changing the schedule")
    args = parser.parse_args()
    config = load_config(args.config)
    overrides = {key: getattr(args, key) for key in
                 ("seed", "run_id", "device", "data_root", "artifact_root", "num_workers")
                 if getattr(args, key) is not None}
    if args.seed is not None and args.run_id is None:
        overrides["run_id"] = f"seed{args.seed}"
    if args.full or args.smoke_test:
        overrides["smoke_test"] = not args.full
    if args.no_download:
        overrides["download"] = False
    if args.split_file:
        if not isinstance(config, LOFOConfig):
            parser.error("--split-file applies only to LOFO configurations.")
        overrides["split_file"] = args.split_file
    config = replace(config, **overrides)
    config.validate()
    run_experiment(config, resume=args.resume, stop_after_epoch=args.stop_after_epoch)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile $RUNTIME_PACKAGE/evaluate.py
"""Opt-in official clean test evaluation, after the full validation protocol ends."""

import json
from pathlib import Path

import torch
from torch import nn
from torchvision.datasets import CIFAR100

from .dataset import make_loader, make_transforms
from .model import build_model
from .train import compatible_config, run_directory, run_epoch
from .utils import (environment_info, load_checkpoint, save_json, seed_everything,
                    select_device, source_digest, utc_now)


def evaluate_clean_test(config):
    config.validate()
    if config.smoke_test:
        raise ValueError("Official test evaluation is disabled for smoke tests.")
    output_dir = run_directory(config)
    result_path = output_dir / "clean_test.json"
    last = load_checkpoint(output_dir / "checkpoints" / "last_checkpoint.pt")
    best_path = output_dir / "checkpoints" / "best_model.pt"
    best = load_checkpoint(best_path)
    if last["epoch"] != config.epochs or len(last["history"]) != config.epochs:
        raise ValueError("Finish the full planned schedule before accessing the test set.")
    if (not compatible_config(last["config"], config.to_dict())
            or not compatible_config(best["config"], config.to_dict())
            or best["epoch"] != last["best_epoch"]
            or best["source_sha256"] != source_digest()):
        raise ValueError("Final checkpoint/configuration mismatch; use the original frozen run.")
    if result_path.exists():
        result = json.loads(result_path.read_text())
        if (result["source_sha256"] != best["source_sha256"]
                or result["selected_epoch"] != best["epoch"]
                or result["split_sha256"] != best["data_manifest"]["split_sha256"]):
            raise ValueError("Saved test result does not match the selected checkpoint.")
        print("Returning the saved final test result; no repeated test-set access.")
        return result
    seed_everything(config.seed, config.deterministic, config.cpu_threads)
    device = select_device()
    model = build_model().to(device)
    model.load_state_dict(best["model_state_dict"])
    _, transform = make_transforms(config)
    # This is the ONLY train=False dataset construction in the implementation.
    try:
        test_set = CIFAR100(config.data_root, train=False, transform=transform,
                            download=config.download)
    except (OSError, RuntimeError) as exc:
        raise RuntimeError("Cannot load official CIFAR-100 test data. Check data_root, "
                           "internet access, and free disk space.") from exc
    if len(test_set) != 10000:
        raise ValueError("Expected all 10,000 official clean test images.")
    loader = make_loader(test_set, config, device,
                         torch.Generator().manual_seed(config.seed + 2))
    # Full precision for the final clean test; selection still uses validation only.
    metrics = run_epoch(model, loader, nn.CrossEntropyLoss(), device, amp=False)
    result = {
        "dataset": "CIFAR100_official_clean_test", "evaluated_utc": utc_now(),
        "test_accuracy_pct": metrics["accuracy_pct"], "test_loss": metrics["loss"],
        "test_count": metrics["count"], "selected_epoch": best["epoch"],
        "selected_validation_accuracy_pct": best["best_validation_accuracy_pct"],
        "selection": config.selection, "checkpoint": str(best_path),
        "source_sha256": best["source_sha256"], "split_sha256": best["data_manifest"]["split_sha256"],
        "evaluation_precision": "float32", "environment": environment_info(device),
        "configuration_frozen": True, "used_for_tuning": False,
    }
    save_json(result_path, result)
    summary_path = output_dir / "summary.json"
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        summary["test_evaluated"] = True
        save_json(summary_path, summary)
    print(f"Final official clean test: accuracy {metrics['accuracy_pct']:.2f}%, "
          f"loss {metrics['loss']:.4f}. Do not tune the configuration using this result.")
    return result


In [ ]:
%%writefile $RUNTIME_ASSETS/NOTICE.md
# CIFAR corruption reference

The selected corruption/helper functions in `cifar_c_reference.py` are extracted
from Dan Hendrycks and Thomas Dietterich's `hendrycks/robustness` repository,
`ImageNet-C/create_c/make_cifar_c.py`, commit
`8190fe329f5f072a06e3a2aea02bb5dda69aed9f` (Apache-2.0; full license alongside).
Source: https://github.com/hendrycks/robustness/blob/8190fe329f5f072a06e3a2aea02bb5dda69aed9f/ImageNet-C/create_c/make_cifar_c.py

Modifications to the extracted reference: unrelated imports/functions and all
dataset-loading/generation code removed; the removed NumPy alias `np.float_`
replaced by its equivalent `np.float64` for NumPy 2 compatibility. No benchmark
images are included. The reference is used only by operator equivalence tests.

`src/corruptions.py` adapts these operators with explicit RNG injection, checked
32×32 RGB PIL inputs/outputs and uint8 serialization. Gaussian/defocus/fog/JPEG
parameters and formulas follow this CIFAR-specific reference. The original
diamond-square helper credits https://github.com/FLHerne/mapgen/blob/master/diamondsquare.py.
Training-time on-the-fly samples are not claimed bit-identical to the released
benchmark: random draws, spatial augmentation, versions and JPEG codecs differ.


In [ ]:
%%writefile $RUNTIME_ASSETS/robustness-LICENSE.txt
                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright [yyyy] [name of copyright owner]

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.



## Inspect and freeze the effective configuration

The saved `config.json`, environment metadata, split indices, source snapshot,
model description, and checkpoint states make the run auditable. Python, NumPy,
PyTorch, CUDA, and loader workers are seeded. Deterministic algorithms are required;
cuDNN benchmarking and TF32 are disabled. CUDA AMP is enabled on supported GPUs
(compute capability ≥ 7); MPS/CPU use float32. Device order is CUDA → MPS → CPU.

Exact replay is expected only within an unchanged software/hardware environment.
PyTorch does not guarantee identical results across releases, CPU/GPU, or GPU models.
Changing precision or runtime on resume emits a warning and records a new session.
Epoch timing includes loading, train and validation work, and excludes persistence,
setup, plotting, and downtime. All accuracy fields use **percent** (0–100).


In [ ]:
from src.config import Config
from src.utils import seed_everything, select_device
from src.model import build_model
from src.train import run_experiment, plot_history, run_directory
import json

config = Config(
    smoke_test=SMOKE_TEST, seed=SEED, split_seed=SPLIT_SEED,
    data_root=str(DATA_ROOT), artifact_root=str(ARTIFACT_ROOT), run_id=RUN_ID, **SETTINGS,
)
config.validate()
print(json.dumps({**config.to_dict(), "effective_epochs": config.epochs,
                  "output_directory": str(run_directory(config))}, indent=2))
# Lightweight architecture check. Training reseeds before constructing its own model.
seed_everything(config.seed, config.deterministic, config.cpu_threads)
check_model = build_model().eval()
with torch.inference_mode():
    check_logits = check_model(torch.zeros(2, 3, 32, 32))
assert tuple(check_logits.shape) == (2, 100), check_logits.shape
assert check_model.conv1.kernel_size == (3, 3) and check_model.conv1.stride == (1, 1)
assert isinstance(check_model.maxpool, torch.nn.Identity)
print("Architecture check passed: [2, 3, 32, 32] → [2, 100]")
del check_model, check_logits


## Train and select using validation only

This cell downloads **official CIFAR-100** via torchvision if needed, creates or
validates the saved 45,000/5,000 split (450/50 per class), and trains the selected
mode. Smoke subsets are drawn only from their respective sides of that split.
The official archive includes the clean test file, but this cell never constructs
or evaluates a test dataset. No corruption benchmark is accessed.

The first epoch with the highest validation accuracy becomes `best_model.pt`.
`last_checkpoint.pt` includes optimizer, cosine scheduler, AMP scaler, RNG and
loader states, configuration, and history. Interruptions replay the incomplete
epoch. Resume never resets or shortens the original learning-rate schedule.
Keep the same configuration and run ID, set `RESUME=True`, and rerun all cells.


In [ ]:
output_dir, history = run_experiment(config, resume=RESUME)
print("Saved artifacts:", output_dir)
print("SMOKE TEST ONLY — not a scientific result" if SMOKE_TEST else "Full schedule completed; validation selection is fixed.")


In [ ]:
# Readable 300-DPI PNGs are already saved; this also displays the plots inline.
# Later figures can be regenerated from metrics.json without retraining.
saved_history = json.loads((output_dir / "metrics.json").read_text())
plot_history(saved_history, output_dir / "plots", config.smoke_test, show=True)
print(json.dumps(json.loads((output_dir / "summary.json").read_text()), indent=2))


## Final official clean test — explicit opt-in, after full training only

Leave this off during development. After the full schedule finishes and model
selection/settings are fixed, set `RUN_FINAL_TEST=True` and run **this cell**.
The function rejects smoke or incomplete runs and loads the validation-selected
best checkpoint. It evaluates all 10,000 official clean test images in float32 and
saves `clean_test.json`; subsequent calls return the stored result. Never use the
test result to alter the selected configuration or select among seeds.

This task does not run this cell against the test set, and never downloads or
evaluates CIFAR-100-C. Future corruption training must augment only training
images. Future LOFO/AugMix conditions must reuse this architecture, saved split,
normalization, selection rule, and comparable optimization budgets.


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    from src.evaluate import evaluate_clean_test
    clean_test_result = evaluate_clean_test(config)
    print(json.dumps(clean_test_result, indent=2))
else:
    print("Official test evaluation not requested. No test images were evaluated.")


## Artifacts and troubleshooting

`ARTIFACT_ROOT/splits/cifar100_seed42.json` is shared across conditions.
Each run lives at `ARTIFACT_ROOT/results/{smoke_test|clean_baseline}/RUN_ID/`:
`config.json`, `environment.json`, `sessions.json`, `protocol.json`, `model.txt`,
`split.json`, `data_manifest.json`, `source/`, `metrics.csv`, `metrics.json`,
`summary.json`, `plots/*.png`, and `checkpoints/{best_model.pt,last_checkpoint.pt}`.
All smoke outputs live beneath `smoke_test` and plot titles/metric rows label them.

If download fails, check internet/disk space or set `DATA_ROOT` to a prepared
torchvision cache and `download=False`. If workers fail in a local interactive
environment, use `num_workers=0` **before starting a fresh run**. If GPU memory is
insufficient, restart with a smaller batch size and a new run ID; record this as a
changed configuration. Enable Drive before long training. Allow roughly 1 GB of
free space for dataset extraction and checkpoints. Keep environment versions fixed
for later conditions; do not silently resume with changed code or settings.

References: [CIFAR dataset](https://www.cs.toronto.edu/~kriz/cifar.html),
[torchvision ResNet18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html),
[PyTorch reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html),
[AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples.html).
